# 자율성의 대가

같은 최종 답에 도달한 두 합성 궤적을 비용·지연·경로 편차·디버깅 단서로 비교합니다.

## 실행 환경 준비

사건별 시간과 비용 단위는 교육용 합성값입니다. 제공자 가격이나 Ch 20의 실제 호출 측정을 대신하지 않습니다.

In [1]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
from pathlib import Path
import sys
from typing import TypedDict

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part5/ch17-agent-overview").is_dir())
LAB = ROOT / "labs/part5/ch17-agent-overview"
print("Python:", sys.version.split()[0], "| 실습 폴더:", LAB.relative_to(ROOT))

# 비용·지연 사건의 필드 타입을 고정해 합산 대상을 분명히 합니다.
class CostEvent(TypedDict):
    event: str
    ms: int
    cost: int

Python: 3.12.13 | 실습 폴더: labs/part5/ch17-agent-overview


## 합성 궤적 준비

워크플로우는 경로가 고정되어 있고, 에이전트 후보는 관찰에 따라 검색을 반복했다고 가정합니다.

In [2]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
traces: dict[str, list[CostEvent]] = {
    "workflow": [
        {"event": "retrieve", "ms": 240, "cost": 0},
        {"event": "generate", "ms": 900, "cost": 2},
        {"event": "validate", "ms": 30, "cost": 0},
    ],
    "agent_candidate": [
        {"event": "plan", "ms": 650, "cost": 1},
        {"event": "search", "ms": 380, "cost": 0},
        {"event": "inspect", "ms": 720, "cost": 1},
        {"event": "search", "ms": 410, "cost": 0},
        {"event": "finish", "ms": 830, "cost": 2},
    ],
}
print(*traces.items(), sep="\n")

('workflow', [{'event': 'retrieve', 'ms': 240, 'cost': 0}, {'event': 'generate', 'ms': 900, 'cost': 2}, {'event': 'validate', 'ms': 30, 'cost': 0}])
('agent_candidate', [{'event': 'plan', 'ms': 650, 'cost': 1}, {'event': 'search', 'ms': 380, 'cost': 0}, {'event': 'inspect', 'ms': 720, 'cost': 1}, {'event': 'search', 'ms': 410, 'cost': 0}, {'event': 'finish', 'ms': 830, 'cost': 2}])


## 단계·비용·지연 계산

최종 답의 정확도는 주어지지 않았으므로 처리량 지표만 계산합니다.

In [3]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
def metrics(events: list[CostEvent]) -> dict[str, int]:
    return {
        "steps": len(events),
        "latency_ms": sum(item["ms"] for item in events),
        "cost_units": sum(item["cost"] for item in events),
        "searches": sum(item["event"] == "search" for item in events),
    }

for name, events in traces.items():
    print(name, metrics(events))
assert metrics(traces["workflow"])["latency_ms"] == 1170
assert metrics(traces["agent_candidate"])["steps"] == 5

workflow {'steps': 3, 'latency_ms': 1170, 'cost_units': 2, 'searches': 0}
agent_candidate {'steps': 5, 'latency_ms': 2990, 'cost_units': 4, 'searches': 2}


## 원인 위치 찾기

지연이 큰 사건과 반복 사건을 표시하면 디버깅의 출발점을 만들 수 있습니다.

In [4]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
for name, events in traces.items():
    slowest = max(events, key=lambda item: item["ms"])
    repeated: list[str] = sorted({item["event"] for item in events if sum(e["event"] == item["event"] for e in events) > 1})
    print(name, "가장 긴 사건:", slowest, "반복:", repeated)

workflow 가장 긴 사건: {'event': 'generate', 'ms': 900, 'cost': 2} 반복: []
agent_candidate 가장 긴 사건: {'event': 'finish', 'ms': 830, 'cost': 2} 반복: ['search']


## 경로 편차 비교

같은 최종 결과라도 중간 경로가 다르면 최종 답만으로 효율과 잘못된 행동을 평가할 수 없습니다.

In [5]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
second_agent_run: list[str] = ["plan", "search", "inspect", "finish"]
first_agent_run: list[str] = [item["event"] for item in traces["agent_candidate"]]
print("첫 경로:", first_agent_run)
print("둘째 경로:", second_agent_run)
print("경로 동일:", first_agent_run == second_agent_run)
print("Ch 29에서는 이런 중간 궤적을 별도 평가합니다.")
assert first_agent_run != second_agent_run

첫 경로: ['plan', 'search', 'inspect', 'search', 'finish']
둘째 경로: ['plan', 'search', 'inspect', 'finish']
경로 동일: False
Ch 29에서는 이런 중간 궤적을 별도 평가합니다.
